**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Radar Signal Processing

The applied capstone of the detection-and-estimation arc: pulse compression (resolution without megawatts), Doppler processing (velocity from phase), CFAR detection (thresholds that adapt to the scene), and a taste of SAR. We build a complete pulse-Doppler radar in NumPy and verify every extracted target parameter against the planted truth.

## 1. Pre-requisites

[Statistical SP](./Statistical_Signal_Processing.ipynb) S4 (matched filters/ROC), [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb) (chirps, FFT), [Estimation Theory](../Intro_Math/Estimation_Theory/Estimation_Theory.ipynb).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)

c_light = 3e8
fs, B, T_pulse = 20e6, 5e6, 20e-6                 # 20 MHz sampling, 5 MHz chirp, 20 µs pulse
fc, PRF = 3e9, 5000                                # S-band, 5 kHz pulse rate
t_p = np.arange(0, T_pulse, 1/fs)
chirp_tx = np.exp(1j*np.pi*(B/T_pulse)*(t_p - T_pulse/2)**2)   # LFM pulse

---
### 🕐 Session 1 of 4 — *Pulse Compression* (~40 min)
**Goal:** long pulse in, sharp spike out: bandwidth (not duration) sets resolution.
**Builds on:** [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 2 (Doppler).

---

## 2. The Chirp's Bargain

💡 **Intuition.** Range resolution wants a *short* pulse; detection range wants *energy* (a long pulse). The chirp takes both: transmit long-and-swept, then **matched-filter** on receive — the output collapses to a spike of width $1/B$, as if you'd transmitted an impossibly powerful short pulse. Resolution comes from **bandwidth**, not duration: $\Delta R = c/2B$. The compression gain is the time–bandwidth product $BT$ — here ×100.

In [ ]:
# two targets 75 m apart — the RAW 20 µs pulse spans 3 km of range; compression resolves them

# YOUR CODE HERE


**What just happened.** Both targets recovered at **3000 and 3075 m**, matching the planted truth exactly to the printed precision — from a pulse whose raw extent covers 3000 m of range.

That last point is the one worth dwelling on. The transmitted pulse is 20 µs long, which occupies $cT/2 = 3000$ m of range. Two echoes 75 m apart overlap almost completely on receive; nothing in the raw data looks like two targets. Matched filtering collapses each echo to a spike of width $c/2B = 30$ m, and 75 m separation then resolves comfortably.

**Resolution comes from bandwidth, not duration.** This is the central fact of the session, and the two numbers make it concrete: raw extent 3000 m, compressed resolution 30 m, a factor of **100**. That factor is exactly the time–bandwidth product $BT = 5\,\text{MHz} \times 20\,\mu\text{s} = 100$. The compression gain is not tunable — it is $BT$, and knowing that lets you design a pulse from requirements rather than by trial.

The bargain is worth stating plainly. Detection range needs *energy*, and energy is power × duration, so it wants a long pulse. Range resolution needs echoes not to overlap, so it wants a short one. Peak transmitter power is fixed by hardware. The chirp escapes the trade by sweeping frequency across a long pulse: you transmit 20 µs of energy and receive the resolution of a 0.2 µs pulse. That is why every modern radar transmits chirps.

**And the processing is a theorem you already proved.** `np.correlate(rx, chirp_tx, "valid")` is the matched filter, which maximises output SNR in white noise — the result from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. Radar is its cleanest application: you know precisely what you transmitted, so you know precisely what to correlate against.

**Two things to notice on the plot.** The dashed truth markers sit on the peaks, and the `assert` enforces agreement to better than one resolution cell — so this is a checked claim, not a visual impression. And look below the peaks: the compressed pulse has **range sidelobes**, around −13 dB for an unwindowed chirp. A strong target's sidelobes can bury a weak target nearby, which is why production radars window the reference chirp, accepting a wider main lobe to push the sidelobes down. Same trade as windowing anywhere else in [Foundations 1](./Foundations_of_Signal_Processing_1.ipynb).

Try moving the targets to 20 m apart and re-running: they merge into a single peak, and no processing recovers them. Below $c/2B$, the information is not there.

---
### 🕐 Session 2 of 4 — *Doppler Processing* (~40 min)
**Goal:** velocity from pulse-to-pulse phase: the range-Doppler map, verified against planted targets.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (CFAR).

---

## 3. Velocity Is a Phase Story

💡 **Intuition.** A single pulse can't measure velocity — but across pulses, a moving target's range change of millimeters shifts the echo's *phase* by $4\pi v T_{PRI}/\lambda$ per pulse. Stack $N$ pulses as rows, and each range cell holds a slow-time sinusoid whose frequency IS the Doppler: an **FFT down each column** turns the pile into a range–Doppler map. Stationary clutter piles up at 0 Hz where a notch removes it — the reason pulse-Doppler radar sees a moving car against a mountain.

In [ ]:
# 64-pulse coherent interval; targets: (3000 m, +30 m/s), (5000 m, −15 m/s), clutter at 4000 m
# compress each pulse, then FFT across pulses
# ORACLE: extract peaks (excluding the v≈0 clutter line) and compare to planted truth

# YOUR CODE HERE


**What just happened.** Both movers extracted: **3000 m at +31.2 m/s** and **5002 m at −15.6 m/s**, against planted values of 3000 m / +30 m/s and 5000 m / −15 m/s. And the apparent errors are not estimation error at all — they are the grid.

**Work the resolution out and the "errors" disappear.** The velocity bin width is
$$\Delta v = \frac{\lambda}{2 N T_{PRI}} = \frac{0.1}{2 \cdot 64 \cdot 200\,\mu\text{s}} = 3.906 \text{ m/s}.$$
The planted +30 m/s falls at bin index $30/3.906 = 7.68$, so the nearest bin is 8, at **+31.25 m/s**. The planted −15 sits at index −3.84, nearest bin −4, at **−15.625 m/s**. Those are exactly the printed values. The estimator did not approximate anything — it returned the correct bin, and the offset is quantisation of a continuous velocity onto a discrete FFT grid.

The same holds in range: samples are spaced $c/2f_s = 7.5$ m apart, so 5002 and 5000 are the same range cell. **Both estimates are exact to the resolution of the measurement**, which is a much stronger statement than "close to the truth."

**Why velocity is a phase story.** At 30 m/s and a 5 kHz PRF, the target moves 6 mm between pulses. The range bin is 7.5 m, so in *range* that motion is undetectable — not merely small, but a thousandth of a cell. But the wavelength is 10 cm, and 6 mm of range change is 12 mm of two-way path, which is over 40° of phase. Range is blind to the motion; phase is shouting about it. Coherent radar exists to exploit exactly that asymmetry.

Stack the pulses and each range cell, read down the column, holds a slow-time sinusoid whose frequency *is* the Doppler shift. So velocity estimation is spectral estimation, and `np.fft.fft(comp, axis=0)` is the whole of it. Two time axes — fast time along a row for range, slow time down a column for velocity — and one familiar transform.

**The clutter ridge is the practical payoff.** The stationary return at 4000 m is 6× the amplitude of either target, and in *range alone* it is inseparable from anything at the same distance. In the Doppler dimension it collapses onto the $v = 0$ line while the movers sit at ±4 bins away, so a notch at zero velocity removes it entirely. That is why a pulse-Doppler radar can track a car driving in front of a mountain, and why real ground clutter — often 60 dB above the target — is a solvable problem rather than a fatal one. Note the extraction code does exactly this: `mask = np.abs(vel_axis)[:, None] > 3` excludes the zero-Doppler line before searching for peaks.

**The cost of resolution.** Finer velocity resolution means larger $N$, and $N T_{PRI}$ is the coherent processing interval — 12.8 ms here. Throughout that dwell the target must stay in its range cell and not accelerate appreciably, or the slow-time sinusoid smears. Resolution is bought with dwell time, the same time–frequency trade as every spectral estimate in this curriculum, now with a physical constraint attached to it.

---
### 🕐 Session 3 of 4 — *CFAR Detection* (~35 min)
**Goal:** thresholds that ride the local noise: constant false-alarm rate in inhomogeneous scenes.
**Builds on:** Session 2; [Statistical SP](./Statistical_Signal_Processing.ipynb) S4. &nbsp; **Feeds into:** Session 4 (SAR at a glance).

---

## 4. The Adaptive Threshold

💡 **Intuition.** A fixed threshold fails in real scenes: set for the quiet region and the clutter region floods you with false alarms; set for clutter and you miss everything quiet. **CA-CFAR** estimates the local noise from a sliding window of *reference cells* around each cell under test (excluding *guard cells* so the target doesn't poison its own estimate) and thresholds at a multiple chosen for a fixed false-alarm rate. The threshold *rides the terrain*.

In [ ]:
# 1-D range profile with a noise step (quiet region | clutter region) and 3 targets

# YOUR CODE HERE


**What just happened.** All three targets found, with **1** false alarm from CFAR against **109** from the fixed threshold — across a scene whose noise level steps by 9× at cell 400.

**Read the fixed threshold's failure precisely.** It was not set badly; it was set *correctly for the quiet region*, using `profile[:400].mean()`. It then produced 109 false alarms in the clutter region, because a threshold calibrated for one noise level is meaningless at another. And the dilemma has no fixed-number solution: raise it to survive the clutter and the quiet-zone targets at cells 120 and 300 disappear. **No single threshold works on an inhomogeneous scene**, which is the entire argument for the session.

CFAR escapes by estimating the noise *locally* — a sliding window of reference cells around each cell under test — and thresholding at a multiple of that estimate. The red trace visibly steps up at cell 400, tracking the terrain. The name is worth taking literally: the goal is not a constant threshold but a constant false-alarm *rate*, achieved by normalising away the local noise level so the detection statistic has the same distribution everywhere.

**The guard cells are the subtle part.** After pulse compression a target occupies several adjacent cells. Without a gap between the cell under test and the reference window, the target's own energy would enter its noise estimate, raise its own threshold, and help conceal it — self-masking. `n_guard=2` is what prevents that, and it is a genuine design parameter rather than defensive padding.

**And CFAR's own false alarm is not a defect.** One detection at cell 60 with no target there is exactly what "constant false alarm *rate*" promises: with `scale=9.0` against exponential noise, the per-cell false-alarm probability is small but nonzero, and across ~770 tested cells you should expect of order one. Reporting zero would actually be suspicious — it would mean the threshold was set far too high and detections were being lost. The multiplier is the knob that trades detections against false alarms, which is the ROC curve from [Statistical SP](./Statistical_Signal_Processing.ipynb) S4 compressed into a single constant.

**Two failure modes worth knowing, both live in this demo.** CA-CFAR *averages* its reference cells, so a second target inside the reference window inflates the noise estimate and can mask the first — the reason GO/SO-CFAR and ordered-statistic CFAR exist, using max, min, or a percentile instead of a mean. And at a **clutter edge**, a window straddling the boundary averages two different noise levels and misbehaves on both sides; look closely at the threshold trace either side of cell 400 and the transition region is visible.

The generalisable idea: detection is not a comparison against a fixed number, it is *estimate the local normal, then decide*. That structure recurs well outside radar — adaptive thresholding in image processing, local baselining in anomaly detection — wherever "normal" is not constant across the data.

---
### 🕐 Session 4 of 4 — *Synthetic Aperture at a Glance* (~30 min)
**Goal:** how a small antenna on a moving platform becomes a huge one: SAR in one simulation.
**Builds on:** Sessions 1–3.

---

## 5. SAR: The Aperture You Fly

💡 **Intuition.** [Array resolution](./Array_Processing.ipynb) scales with aperture size — so *fly* the aperture: a plane records echoes along its path, and coherent processing of that kilometer of positions synthesizes a kilometer-wide antenna. The signal along the track is (once again) a **chirp** — quadratic range migration makes phase quadratic in position — so azimuth compression is Session 1's matched filter, rotated 90°. Range chirp + azimuth chirp = imagery from orbit.

In [ ]:
# strip-map SAR toy: 3 point scatterers, platform flying past — azimuth compression
# azimuth matched filter: the reference chirp for a scatterer at x=0

# YOUR CODE HERE


**What just happened.** Three scatterers recovered at **−40.0, 0.0, 35.2 m** in azimuth against planted values of −40, 0, 35 — resolved to a couple of metres at a range of 5 km, using an antenna the simulation never even specifies. The resolution came from the 205 m of *flight path*, not from any physical aperture.

**Why the along-track signal is a chirp.** As the platform passes a scatterer, the instantaneous range is $R(u) = \sqrt{R_0^2 + (u - x_0)^2}$, which near closest approach is approximately quadratic in $u$. Phase is $-4\pi R/\lambda$, so the recorded phase is quadratic in *position* — and quadratic phase is exactly what a chirp is. So azimuth compression is Session 1's matched filter applied along the track: `np.correlate(az_sig, h_az, "same")` is the same line of code, rotated ninety degrees. Range chirp compresses range; azimuth chirp compresses azimuth; multiply the two and you have an image.

Notice also that `h_az` is built for a scatterer at $x = 0$, yet it compresses all three. The chirp's shape depends on range, not on azimuth position, so one reference filter serves the whole line — which is precisely why the correlation works.

**The counterintuitive payoff.** Ask what happens to azimuth resolution as the platform flies *further from* the scene. The instinct is that it degrades. It does not: at greater range the target stays inside the real antenna's beam for a longer stretch of flight, so the synthetic aperture grows in proportion, and the two effects very nearly cancel. Strip-map SAR azimuth resolution is approximately $D/2$ — **half the physical antenna length**, independent of range.

Which yields a genuinely strange design rule: a *smaller* antenna gives *better* azimuth resolution, because a smaller antenna has a wider beam, which keeps each target illuminated over a longer synthetic aperture. That is the opposite of every optical intuition, and it is why radar satellites image the Earth at metre resolution from hundreds of kilometres up.

**What makes it fragile.** The whole method rests on phase coherence across positions. Platform position errors, oscillator drift, or atmospheric path variation destroy the phase relationship, and the synthetic aperture collapses back to the physical one. This is why SAR platforms carry precision navigation and why autofocus algorithms are standard — it is the same coherence requirement as Session 2's Doppler processing, transposed onto a spatial axis.

**And the honest scope of this demo.** It compresses in azimuth only, at a single range, with no range cell migration correction, no range–azimuth coupling, and no motion errors. Real SAR processors — range–Doppler, chirp scaling, back-projection — exist almost entirely to handle those. This cell shows the principle that makes SAR possible; the engineering that makes it work is a field in itself.

**The workshop in one line.** Bandwidth buys range resolution, pulse-to-pulse phase buys velocity, local noise estimation buys detection, and motion buys aperture — with the matched filter doing the work in three of the four.

## 6. Conclusion

Bandwidth buys range resolution (targets 40 m apart, resolved and verified); pulse-to-pulse phase buys velocity (both movers extracted to the planted values); CFAR buys detection that survives real scenes (1 false alarm vs the fixed threshold's 109, across a 9× noise step — a constant *rate*, as the name promises); and motion buys aperture. One curriculum's worth of tools, pointed at the sky.

---
## Where next

- [Array Processing](./Array_Processing.ipynb) — real apertures; [MIMO Communications](./MIMO_Communications.ipynb) — the comms twin.
- [Software-Defined Radio](../Intro_SDR/Software_Defined_Radio.ipynb) — passive radar with a $30 dongle is a real (advanced) project.